In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
 'tinker', 'tinker-cookbook', 'av', 'huggingface_hub'], check=True)

# ── Keys ─────────────────────────────────────────────────────────────────────
import os, re, json, asyncio, shutil, zipfile, urllib.request
from pathlib import Path
from io import BytesIO

os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY_HERE'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'
os.environ['HUGGING_FACE_HUB_TOKEN'] = 'YOUR_HF_TOKEN_HERE'
HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
GITHUB_TOKEN = 'YOUR_GITHUB_PAT_HERE'
GITHUB_RAW = 'https://raw.githubusercontent.com/QuantumByte-01/PhysSim-VLM/main'

# ── Smoke test flag ───────────────────────────────────────────────────────────
SMOKE_TEST = False

def gh_download(url, dest):
 req = urllib.request.Request(url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
 with urllib.request.urlopen(req) as r:
 Path(dest).write_bytes(r.read())

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK = Path('/kaggle/working')
DL_DIR = WORK / 'physbench'
OUT_DIR = WORK / 'results' / 'physbench_grpo_r3_test'
PRED_FILE = OUT_DIR / 'predictions.json'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Helper ────────────────────────────────────────────────────────────────────
def find_dir(root, *names):
 for name in names:
 for p in Path(root).rglob(name):
 if p.is_dir(): return p
 return None

# ── Extract zips if needed ────────────────────────────────────────────────────
for zip_name, dir_names in [('video.zip', ('video', 'videos')), ('image.zip', ('image', 'images'))]:
 for zp in DL_DIR.rglob(zip_name):
 if not find_dir(DL_DIR, *dir_names):
 print(f'Extracting {zp}...')
 with zipfile.ZipFile(zp) as z:
 z.extractall(zp.parent)
 print(f' Done.')
 break

# ── Find media dirs - fall back to DL_DIR if files are flat ──────────────────
IMG_DIR = find_dir(DL_DIR, 'image', 'images', 'img') or DL_DIR
VID_DIR = find_dir(DL_DIR, 'video', 'videos', 'vid') or DL_DIR

# ── Find JSON files ───────────────────────────────────────────────────────────
all_jsons = list(DL_DIR.rglob('*.json'))
test_json = next((j for j in all_jsons if j.name == 'test.json'), None)
test_answer = next((j for j in all_jsons if j.name == 'test_answer.json'), None)
if not test_answer:
 test_answer = next((j for j in all_jsons if 'answer' in j.name.lower() and 'test' in j.name.lower()), None)
if not test_answer:
 test_answer = next((j for j in all_jsons if 'answer' in j.name.lower()), None)

if test_json is None:
 dest = DL_DIR / 'test.json'
 gh_download(f'{GITHUB_RAW}/data/raw/physbench/test.json', dest)
 test_json = dest

if test_answer is None:
 dest = DL_DIR / 'test_answer.json'
 gh_download(f'{GITHUB_RAW}/test_answer.json', dest)
 test_answer = dest

PHYSBENCH = test_json.parent if test_json else DL_DIR

print(f'test.json: {test_json}')
print(f'test_answer.json: {test_answer}')
print(f'image dir: {IMG_DIR}')
print(f'video dir: {VID_DIR}')

# ── Download checkpoint from GitHub only if local is behind ──────────────────
def local_count():
 try: return len(json.load(open(PRED_FILE)))
 except: return 0

def github_count():
 try:
 req = urllib.request.Request(
 f'{GITHUB_RAW}/results/physbench_grpo_r3_test/predictions.json',
 headers={'Authorization': f'token {GITHUB_TOKEN}'})
 with urllib.request.urlopen(req) as r:
 return len(json.loads(r.read()))
 except: return 0

local_n = local_count()
github_n = github_count()
print(f'Local checkpoint: {local_n} | GitHub checkpoint: {github_n}')

if github_n > local_n:
 print('GitHub is ahead - downloading...')
 gh_download(f'{GITHUB_RAW}/results/physbench_grpo_r3_test/predictions.json', PRED_FILE)
 print(f'Checkpoint loaded: {local_count()} samples done')
else:
 print(f'Using local checkpoint ({local_n} samples)')

# ── Eval config ───────────────────────────────────────────────────────────────
BASE_MODEL = 'Qwen/Qwen3-VL-30B-A3B-Instruct'
MODEL_PATH = 'tinker://<run-id>:train:0/sampler_weights/final'
CONCURRENCY = 16
VIDEO_FRAMES = 8
MAX_ASSET_BYTES = 1_800_000
MAX_FRAME_DIM = 1120

from PIL import Image
import tinker
from tinker import types
from tinker_cookbook import tokenizer_utils

def _compress_image(img):
 img = img.convert('RGB')
 w, h = img.size
 if max(w, h) > MAX_FRAME_DIM:
 scale = MAX_FRAME_DIM / max(w, h)
 img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
 for quality in (85, 70, 55, 40):
 buf = BytesIO()
 img.save(buf, format='JPEG', quality=quality)
 if buf.tell() <= MAX_ASSET_BYTES:
 return buf.getvalue()
 return buf.getvalue()

def extract_video_frames(video_path, n_frames=VIDEO_FRAMES):
 """Memory-efficient: decode frames one at a time, only keep selected indices as JPEG bytes."""
 import av
 with av.open(str(video_path)) as container:
 stream = container.streams.video[0]
 total = stream.frames or 0
 if total > 0:
 target = set(min(int(i * total / n_frames), total - 1) for i in range(n_frames))
 result = []
 for idx, frame in enumerate(container.decode(video=0)):
 if idx in target:
 result.append(_compress_image(frame.to_image()))
 if idx >= max(target):
 break
 return result
 else:
 # Unknown length - collect all then subsample (unavoidable)
 all_frames = [f.to_image() for f in container.decode(video=0)]
 if not all_frames: return []
 indices = [min(int(i * len(all_frames) / n_frames), len(all_frames) - 1) for i in range(n_frames)]
 return [_compress_image(all_frames[i]) for i in dict.fromkeys(indices)]

def image_to_bytes(path): return _compress_image(Image.open(path))

def build_model_input(tokenizer, image_bytes_list, text):
 chunks = [types.EncodedTextChunk(tokens=tokenizer.encode('<|im_start|>user\n', add_special_tokens=False))]
 vs = tokenizer.encode('<|vision_start|>', add_special_tokens=False)
 ve = tokenizer.encode('<|vision_end|>', add_special_tokens=False)
 for b in image_bytes_list:
 chunks += [types.EncodedTextChunk(tokens=vs), types.ImageChunk(data=b, format='jpeg'), types.EncodedTextChunk(tokens=ve)]
 chunks.append(types.EncodedTextChunk(tokens=tokenizer.encode(text + '<|im_end|>\n<|im_start|>assistant\n', add_special_tokens=False)))
 return types.ModelInput(chunks=chunks)

def load_test_split():
 questions = {q['idx']: q for q in json.load(open(test_json)) if q['split'] == 'test'}
 answers = {a['idx']: a for a in json.load(open(test_answer)) if a.get('idx') in questions}
 return [{'idx': idx, 'mode': q['mode'], 'task_type': answers[idx].get('task_type','unknown'),
 'sub_type': answers[idx].get('sub_type','unknown'), 'ability_type': answers[idx].get('ability_type','unknown'),
 'question': q['question'], 'file_names': q['file_name'], 'answer': answers[idx]['answer'].strip().upper()}
 for idx, q in questions.items() if idx in answers and answers[idx].get('answer')]

def collect_image_bytes(record):
 result = []
 for fname in record['file_names']:
 if fname.endswith('.mp4'):
 p = VID_DIR / fname
 if not p.exists(): return None
 try: result.extend(extract_video_frames(p))
 except: return None
 else:
 found = False
 for d in [IMG_DIR, PHYSBENCH]:
 p = d / fname
 if p.exists():
 try: result.append(image_to_bytes(p)); found = True; break
 except: return None
 if not found: return None
 return result

def extract_choice(text):
 if not text or text.startswith('ERROR'): return ''
 t = text.upper()
 for pat in [r'ANSWER[:\s]+([A-D])', r'THEREFORE[,\s]+([A-D])\b', r'OPTION\s+([A-D])\b',
 r'CORRECT.*?([A-D])\b', r'\bANSWER IS\s+([A-D])\b']:
 m = re.search(pat, t)
 if m: return m.group(1)
 matches = list(re.finditer(r'\b([A-D])\b', t))
 return matches[-1].group(1) if matches else ''

async def predict_async(client, tokenizer, record):
 loop = asyncio.get_event_loop()
 img_bytes = await loop.run_in_executor(None, collect_image_bytes, record)
 if img_bytes is None: return None
 prompt = build_model_input(tokenizer, img_bytes, re.sub(r'<video>|<image>', '', record['question']).strip())
 try:
 r = await client.sample_async(prompt=prompt, num_samples=1,
 sampling_params=types.SamplingParams(max_tokens=512, temperature=0.0, stop=['<|im_end|>']))
 return tokenizer.decode(r.sequences[0].tokens, skip_special_tokens=True).strip()
 except Exception as e:
 return f'ERROR: {e}'

async def run_eval(client, tokenizer, records):
 done = {p['idx']: p for p in json.load(open(PRED_FILE))} if PRED_FILE.exists() else {}
 print(f'Resuming: {len(done)} already done')
 predictions = list(done.values())
 remaining = [r for r in records if r['idx'] not in done]
 if SMOKE_TEST:
 remaining = remaining[:5]
 print(f'SMOKE TEST: running {len(remaining)} samples only (not saving to checkpoint)')
 else:
 print(f'Running {len(remaining)} samples (concurrency={CONCURRENCY})...')
 sem, lock, count = asyncio.Semaphore(CONCURRENCY), asyncio.Lock(), 0

 async def process(record):
 nonlocal count
 async with sem:
 raw = await predict_async(client, tokenizer, record)
 choice = extract_choice(raw or '')
 result = {'idx': record['idx'], 'mode': record['mode'], 'task_type': record['task_type'],
 'sub_type': record['sub_type'], 'ability_type': record['ability_type'],
 'answer': record['answer'], 'predicted': choice, 'raw': (raw or '')[:800],
 'correct': choice == record['answer']}
 async with lock:
 count += 1
 tick = 'OK' if result['correct'] else 'XX'
 print(f'[{count:4}/{len(remaining)}] idx={record["idx"]:5} mode={record["mode"]:<12} GT={record["answer"]} pred={choice} {tick}', flush=True)
 if not SMOKE_TEST:
 predictions.append(result)
 if count % 50 == 0:
 json.dump(predictions, open(PRED_FILE, 'w'), indent=2)

 await asyncio.gather(*[process(r) for r in remaining])
 if not SMOKE_TEST:
 json.dump(predictions, open(PRED_FILE, 'w'), indent=2)
 return predictions

# ── Run ───────────────────────────────────────────────────────────────────────
print('\nLoading tokenizer...')
tokenizer = tokenizer_utils.get_tokenizer(BASE_MODEL)
svc = tinker.ServiceClient()
client = svc.create_sampling_client(model_path=MODEL_PATH)
records = load_test_split()
print(f'Total records: {len(records)}')

predictions = await run_eval(client, tokenizer, records)

if not SMOKE_TEST:
 total = len(predictions)
 correct = sum(1 for p in predictions if p['correct'])
 print(f'\nFINAL: {correct}/{total} = {100*correct/total:.2f}%')
 print(f'Saved to {PRED_FILE}')
else:
 print('\nSmoke test done. Set SMOKE_TEST = False to run full eval.')